# Opta Shot Qualifier Audit

Scans every JSONP match file under `data/raw/PRD/2025-2026/matches/` and collects
every qualifier that appears on shot-type events (typeIds 13, 14, 15, 16).

**Goal:** Find qualifiers that appear on shots but are NOT in `data/mapping/opta-qualifiers.js`.

**Shot typeIds:**
- `13` — Miss (shot wide or over)
- `14` — Post (hits woodwork)
- `15` — Attempt Saved
- `16` — Goal

In [1]:
import json
import re
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd

ROOT        = Path("..")
MATCHES_DIR = ROOT / "data" / "raw" / "PRD" / "2025-2026" / "matches"
QUALS_JS    = ROOT / "data" / "mapping" / "opta-qualifiers.js"

SHOT_TYPE_IDS = {13, 14, 15, 16}
SHOT_TYPE_NAMES = {
    13: "Miss",
    14: "Post",
    15: "Attempt Saved",
    16: "Goal",
}

print(f"Matches dir    : {MATCHES_DIR.resolve()}")
print(f"  exists       : {MATCHES_DIR.exists()}")
print(f"Qualifiers JS  : {QUALS_JS.resolve()}")
print(f"  exists       : {QUALS_JS.exists()}")

Matches dir    : C:\Users\Usuario\OneDrive\football-analytics-research\data\raw\PRD\2025-2026\matches
  exists       : True
Qualifiers JS  : C:\Users\Usuario\OneDrive\football-analytics-research\data\mapping\opta-qualifiers.js
  exists       : True


In [2]:
# ── Load qualifier mapping from opta-qualifiers.js ───────────────────────────

def load_qualifiers_mapping(js_path: Path) -> dict[int, dict]:
    """
    Parse opta-qualifiers.js and return {qualifierId (int): {name, description}}.
    Uses same JS→JSON normalisation as the events mapping loader.
    """
    content = js_path.read_text(encoding="utf-8")

    match = re.search(r'var\s+\w+\s*=\s*(\{.*\});?', content, re.DOTALL)
    if not match:
        raise ValueError(f"Could not extract mapping dict from {js_path}")

    js_obj = match.group(1)
    js_obj = re.sub(r'(\w+):', r'"\1":', js_obj)   # quote all bare keys
    js_obj = re.sub(r"'([^']*?)'", r'"\1"', js_obj) # single → double quotes
    js_obj = re.sub(r',\s*\}', '}', js_obj)          # strip trailing commas

    raw = json.loads(js_obj)
    return {int(k): v for k, v in raw.items()}


def qual_name(qid: int, mapping: dict) -> str:
    if qid not in mapping:
        return "*** NOT IN MAPPING ***"
    entry = mapping[qid]
    return entry.get("name", str(qid)) if isinstance(entry, dict) else str(entry)


quals_mapping = load_qualifiers_mapping(QUALS_JS)
print(f"Loaded {len(quals_mapping)} qualifier entries from opta-qualifiers.js")
print("\nSample entries:")
for qid in [15, 20, 72, 108, 9, 214]:
    print(f"  {qid:>4}  {qual_name(qid, quals_mapping)}")

Loaded 273 qualifier entries from opta-qualifiers.js

Sample entries:
    15  Head
    20  Right footed
    72  Left footed
   108  Volley
     9  Penalty
   214  Big Chance


In [3]:
# ── JSONP helpers ─────────────────────────────────────────────────────────────

def extract_json_from_jsonp(path: Path) -> dict:
    raw = path.read_text(encoding="utf-8")
    start = raw.index("{")
    payload = raw[start:].rstrip()
    if payload.endswith(")"):
        payload = payload[:-1]
    return json.loads(payload)


# ── Scan all match files, collect shot events ─────────────────────────────────

def scan_shots(
    matches_dir: Path,
    shot_type_ids: set[int],
) -> list[dict]:
    """
    Walk every match file and return one dict per shot event with all qualifier IDs.
    """
    shots: list[dict] = []
    errors: list[str] = []

    all_files = sorted(
        p for p in matches_dir.iterdir()
        if p.is_file() and not p.name.startswith(".")
    )
    print(f"Found {len(all_files)} file(s) to scan...")

    for fp in all_files:
        try:
            data   = extract_json_from_jsonp(fp)
            events = data["liveData"]["event"]
        except Exception as exc:
            errors.append(f"{fp.name}: {exc}")
            continue

        for ev in events:
            tid = ev.get("typeId")
            if tid not in shot_type_ids:
                continue

            qualifier_ids = [q.get("qualifierId") for q in ev.get("qualifier", [])]
            qualifier_vals = {
                q.get("qualifierId"): q.get("value")
                for q in ev.get("qualifier", [])
            }

            shots.append({
                "match"         : fp.name,
                "type_id"       : tid,
                "type_name"     : SHOT_TYPE_NAMES.get(tid, str(tid)),
                "period"        : ev.get("periodId"),
                "minute"        : ev.get("timeMin"),
                "outcome"       : ev.get("outcome"),
                "x"             : ev.get("x"),
                "y"             : ev.get("y"),
                "qualifier_ids" : qualifier_ids,
                "qualifier_vals": qualifier_vals,
            })

    print(f"✓ Scanned {len(all_files) - len(errors)} files")
    print(f"✗ Errors  {len(errors)} files")
    if errors:
        for e in errors:
            print(f"  {e}")
    return shots


shots = scan_shots(MATCHES_DIR, SHOT_TYPE_IDS)

print(f"\nTotal shot events collected: {len(shots):,}")
print("\nBreakdown by type:")
for tid, name in SHOT_TYPE_NAMES.items():
    n = sum(1 for s in shots if s["type_id"] == tid)
    print(f"  typeId {tid:>2}  {name:<15}  {n:>5,}")

Found 230 file(s) to scan...
✓ Scanned 230 files
✗ Errors  0 files

Total shot events collected: 5,793

Breakdown by type:
  typeId 13  Miss             2,163
  typeId 14  Post                98
  typeId 15  Attempt Saved    2,927
  typeId 16  Goal               605


In [4]:
# ── Overall qualifier distribution across ALL shot events ─────────────────────

total_shots = len(shots)
all_qual_counts: Counter = Counter()
for s in shots:
    all_qual_counts.update(s["qualifier_ids"])

rows = []
for qid, cnt in all_qual_counts.most_common():
    rows.append({
        "qualifierId"  : qid,
        "name"         : qual_name(qid, quals_mapping),
        "occurrences"  : cnt,
        "pct_of_shots" : f"{cnt / total_shots * 100:.1f}%",
        "in_mapping"   : qid in quals_mapping,
    })

df_all_quals = pd.DataFrame(rows)
print(f"All qualifiers found on {total_shots:,} shot events ({len(all_qual_counts)} distinct qualifier IDs):\n")
display(df_all_quals)

All qualifiers found on 5,793 shot events (103 distinct qualifier IDs):



,qualifierId,name,occurrences,pct_of_shots,in_mapping
0,103,Goal mouth z co-ordinate,5793,100.0%,True
1,102,Goal mouth y co-ordinate,5793,100.0%,True
2,56,Zone,5793,100.0%,True
3,231,GK Y Coordinate,4434,76.5%,True
4,230,GK X Coordinate,4434,76.5%,True
...,...,...,...,...,...
98,358,*** NOT IN MAPPING ***,1,0.0%,False
99,484,*** NOT IN MAPPING ***,1,0.0%,False
100,459,*** NOT IN MAPPING ***,1,0.0%,False
101,490,*** NOT IN MAPPING ***,1,0.0%,False


In [5]:
# ── Qualifiers NOT in the mapping ─────────────────────────────────────────────

unknown = df_all_quals[~df_all_quals["in_mapping"]].copy()

if unknown.empty:
    print("✅  Every qualifier on shots is present in opta-qualifiers.js — nothing missing.")
else:
    print(f"🚨  {len(unknown)} qualifier(s) found on shots that are NOT in opta-qualifiers.js:\n")
    display(unknown)

    # For each unknown qualifier, show a few example events so we can reason about it
    MAX_EXAMPLES = 10
    for _, row in unknown.iterrows():
        qid = row["qualifierId"]
        print(f"\n{'─'*60}")
        print(f"  qualifierId {qid}  |  {row['occurrences']} occurrences on shots")
        print(f"{'─'*60}")

        examples = [
            s for s in shots
            if qid in s["qualifier_ids"]
        ][:MAX_EXAMPLES]

        # Which shot types carry this qualifier?
        type_dist = Counter(
            f"{s['type_id']} ({s['type_name']})"
            for s in shots
            if qid in s["qualifier_ids"]
        )
        print("\n  Shot types that carry this qualifier:")
        for label, cnt in type_dist.most_common():
            print(f"    {cnt:>4}x   {label}")

        # Value distribution (most qualifiers have no value, but some do)
        val_dist = Counter(
            s["qualifier_vals"].get(qid)
            for s in shots
            if qid in s["qualifier_ids"]
        )
        print("\n  Qualifier value distribution:")
        for val, cnt in val_dist.most_common(10):
            print(f"    {cnt:>4}x   {repr(val)}")

        # Co-occurring qualifiers (what else appears alongside this unknown qualifier?)
        cooccur: Counter = Counter()
        for s in shots:
            if qid in s["qualifier_ids"]:
                for other_qid in s["qualifier_ids"]:
                    if other_qid != qid:
                        cooccur[other_qid] += 1
        print("\n  Top co-occurring qualifiers:")
        for other_qid, cnt in cooccur.most_common(10):
            print(f"    {cnt:>4}x   qualifierId {other_qid}  ({qual_name(other_qid, quals_mapping)})")

        # Sample rows
        print(f"\n  Sample shots (up to {MAX_EXAMPLES}):")
        display(
            pd.DataFrame(examples)[[
                "match", "type_name", "period", "minute", "outcome", "x", "y", "qualifier_ids"
            ]]
        )

🚨  20 qualifier(s) found on shots that are NOT in opta-qualifiers.js:



,qualifierId,name,occurrences,pct_of_shots,in_mapping
20,458,*** NOT IN MAPPING ***,1502,25.9%,False
27,375,*** NOT IN MAPPING ***,605,10.4%,False
28,374,*** NOT IN MAPPING ***,605,10.4%,False
29,395,*** NOT IN MAPPING ***,598,10.3%,False
30,396,*** NOT IN MAPPING ***,598,10.3%,False
42,391,*** NOT IN MAPPING ***,255,4.4%,False
43,468,*** NOT IN MAPPING ***,243,4.2%,False
54,472,*** NOT IN MAPPING ***,116,2.0%,False
61,390,*** NOT IN MAPPING ***,85,1.5%,False
62,353,*** NOT IN MAPPING ***,85,1.5%,False



────────────────────────────────────────────────────────────
  qualifierId 458  |  1502 occurrences on shots
────────────────────────────────────────────────────────────

  Shot types that carry this qualifier:
     752x   15 (Attempt Saved)
     538x   13 (Miss)
     184x   16 (Goal)
      28x   14 (Post)

  Qualifier value distribution:
    1502x   None

  Top co-occurring qualifiers:
    1502x   qualifierId 103  (Goal mouth z co-ordinate)
    1502x   qualifierId 102  (Goal mouth y co-ordinate)
    1502x   qualifierId 56  (Zone)
    1204x   qualifierId 231  (GK Y Coordinate)
    1204x   qualifierId 230  (GK X Coordinate)
     870x   qualifierId 146  (Blocked x co-ordinate)
     870x   qualifierId 147  (Blocked y co-ordinate)
     847x   qualifierId 20  (Right footed)
     787x   qualifierId 22  (Regular play)
     724x   qualifierId 233  (Opposite related event ID)

  Sample shots (up to 10):


,match,type_name,period,minute,outcome,x,y,qualifier_ids
0,10d1132abu0fa9xolj05top3o,Miss,1,4,1,77.2,55.8,"[328, 103, 215, 231, 102, 160, 230, 77, 72, 45..."
1,10d1132abu0fa9xolj05top3o,Attempt Saved,1,34,1,74.3,35.8,"[102, 328, 215, 103, 24, 146, 20, 82, 18, 233,..."
2,10d1132abu0fa9xolj05top3o,Attempt Saved,1,47,1,92.9,32.9,"[56, 102, 20, 233, 22, 147, 103, 78, 458, 328,..."
3,10d1132abu0fa9xolj05top3o,Attempt Saved,2,54,1,90.6,39.3,"[22, 15, 102, 328, 233, 56, 147, 458, 146, 103..."
4,10d1132abu0fa9xolj05top3o,Attempt Saved,2,61,1,90.2,48.5,"[146, 458, 147, 108, 103, 231, 233, 25, 17, 10..."
5,10d1132abu0fa9xolj05top3o,Attempt Saved,2,61,1,79.9,41.5,"[231, 102, 146, 458, 147, 215, 82, 230, 18, 56..."
6,10d1132abu0fa9xolj05top3o,Attempt Saved,2,74,1,92.0,42.7,"[146, 15, 214, 147, 102, 56, 22, 233, 328, 17,..."
7,10d1132abu0fa9xolj05top3o,Goal,2,74,1,96.2,54.6,"[20, 472, 16, 328, 214, 395, 396, 22, 458, 281..."
8,10d1132abu0fa9xolj05top3o,Attempt Saved,2,80,1,76.6,31.1,"[18, 102, 146, 26, 103, 56, 458, 78, 147, 120,..."
9,10d1132abu0fa9xolj05top3o,Attempt Saved,2,92,1,96.8,69.5,"[20, 458, 147, 146, 22, 233, 102, 468, 103, 65..."



────────────────────────────────────────────────────────────
  qualifierId 375  |  605 occurrences on shots
────────────────────────────────────────────────────────────

  Shot types that carry this qualifier:
     605x   16 (Goal)

  Qualifier value distribution:
       1x   '74:13.610'
       1x   '63:51.548'
       1x   '76:31.092'
       1x   '89:08.876'
       1x   '08:28.091'
       1x   '10:59.852'
       1x   '41:27.192'
       1x   '60:02.740'
       1x   '09:37.687'
       1x   '33:39.653'

  Top co-occurring qualifiers:
     605x   qualifierId 374  (*** NOT IN MAPPING ***)
     605x   qualifierId 102  (Goal mouth y co-ordinate)
     605x   qualifierId 231  (GK Y Coordinate)
     605x   qualifierId 56  (Zone)
     605x   qualifierId 230  (GK X Coordinate)
     605x   qualifierId 103  (Goal mouth z co-ordinate)
     598x   qualifierId 395  (*** NOT IN MAPPING ***)
     598x   qualifierId 396  (*** NOT IN MAPPING ***)
     406x   qualifierId 29  (Assisted)
     406x   qualifie

...

,match,type_name,period,minute,outcome,x,y,qualifier_ids
0,1xweeu166hhelv9hh9l8lnpjo,Attempt Saved,1,46,1,73.4,35.8,"[146, 147, 459, 233, 76, 29, 215, 24, 103, 490..."



────────────────────────────────────────────────────────────
  qualifierId 490  |  1 occurrences on shots
────────────────────────────────────────────────────────────

  Shot types that carry this qualifier:
       1x   15 (Attempt Saved)

  Qualifier value distribution:
       1x   '17'

  Top co-occurring qualifiers:
       1x   qualifierId 146  (Blocked x co-ordinate)
       1x   qualifierId 147  (Blocked y co-ordinate)
       1x   qualifierId 459  (*** NOT IN MAPPING ***)
       1x   qualifierId 233  (Opposite related event ID)
       1x   qualifierId 76  (Low left)
       1x   qualifierId 29  (Assisted)
       1x   qualifierId 215  (Individual Play)
       1x   qualifierId 24  (Set piece)
       1x   qualifierId 103  (Goal mouth z co-ordinate)
       1x   qualifierId 20  (Right footed)

  Sample shots (up to 10):


,match,type_name,period,minute,outcome,x,y,qualifier_ids
0,1xweeu166hhelv9hh9l8lnpjo,Attempt Saved,1,46,1,73.4,35.8,"[146, 147, 459, 233, 76, 29, 215, 24, 103, 490..."



────────────────────────────────────────────────────────────
  qualifierId 474  |  1 occurrences on shots
────────────────────────────────────────────────────────────

  Shot types that carry this qualifier:
       1x   13 (Miss)

  Qualifier value distribution:
       1x   '465'

  Top co-occurring qualifiers:
       1x   qualifierId 102  (Goal mouth y co-ordinate)
       1x   qualifierId 17  (Box-centre)
       1x   qualifierId 231  (GK Y Coordinate)
       1x   qualifierId 56  (Zone)
       1x   qualifierId 22  (Regular play)
       1x   qualifierId 55  (Related event ID)
       1x   qualifierId 121  (Swerve Right)
       1x   qualifierId 73  (Left)
       1x   qualifierId 72  (Left footed)
       1x   qualifierId 468  (*** NOT IN MAPPING ***)

  Sample shots (up to 10):


,match,type_name,period,minute,outcome,x,y,qualifier_ids
0,1z79bgsg0g4xojqjpoxolh1ck,Miss,2,54,1,87.7,38.9,"[102, 17, 231, 56, 22, 55, 121, 73, 72, 468, 2..."


In [6]:
# ── Per-shot-type qualifier breakdown ─────────────────────────────────────────
# For each shot typeId, show how often each qualifier appears (as % of that shot type).
# This makes it easy to spot qualifiers that cluster on only one type.

for tid, name in SHOT_TYPE_NAMES.items():
    subset = [s for s in shots if s["type_id"] == tid]
    if not subset:
        continue

    n = len(subset)
    qual_counts: Counter = Counter()
    for s in subset:
        qual_counts.update(s["qualifier_ids"])

    print("=" * 60)
    print(f"  typeId {tid} — {name}   ({n:,} events)")
    print("=" * 60)

    rows = []
    for qid, cnt in qual_counts.most_common():
        rows.append({
            "qualifierId": qid,
            "name"       : qual_name(qid, quals_mapping),
            "count"      : cnt,
            "pct"        : f"{cnt / n * 100:.1f}%",
            "in_mapping" : qid in quals_mapping,
        })
    display(pd.DataFrame(rows))
    print()

  typeId 13 — Miss   (2,163 events)


,qualifierId,name,count,pct,in_mapping
0,103,Goal mouth z co-ordinate,2163,100.0%,True
1,231,GK Y Coordinate,2163,100.0%,True
2,102,Goal mouth y co-ordinate,2163,100.0%,True
3,230,GK X Coordinate,2163,100.0%,True
4,56,Zone,2163,100.0%,True
...,...,...,...,...,...
65,71,35+ left,2,0.1%,True
66,233,Opposite related event ID,1,0.0%,True
67,70,35+ right,1,0.0%,True
68,474,*** NOT IN MAPPING ***,1,0.0%,False



  typeId 14 — Post   (98 events)


,qualifierId,name,count,pct,in_mapping
0,102,Goal mouth y co-ordinate,98,100.0%,True
1,231,GK Y Coordinate,98,100.0%,True
2,56,Zone,98,100.0%,True
3,230,GK X Coordinate,98,100.0%,True
4,103,Goal mouth z co-ordinate,98,100.0%,True
5,55,Related event ID,70,71.4%,True
6,29,Assisted,70,71.4%,True
7,22,Regular play,56,57.1%,True
8,20,Right footed,52,53.1%,True
9,328,First Time,47,48.0%,True



  typeId 15 — Attempt Saved   (2,927 events)


,qualifierId,name,count,pct,in_mapping
0,102,Goal mouth y co-ordinate,2927,100.0%,True
1,103,Goal mouth z co-ordinate,2927,100.0%,True
2,146,Blocked x co-ordinate,2927,100.0%,True
3,147,Blocked y co-ordinate,2927,100.0%,True
4,56,Zone,2927,100.0%,True
...,...,...,...,...,...
67,66,Out of box-deep right,2,0.1%,True
68,362,*** NOT IN MAPPING ***,2,0.1%,False
69,21,Other body part,2,0.1%,True
70,459,*** NOT IN MAPPING ***,1,0.0%,False



  typeId 16 — Goal   (605 events)


,qualifierId,name,count,pct,in_mapping
0,375,*** NOT IN MAPPING ***,605,100.0%,False
1,374,*** NOT IN MAPPING ***,605,100.0%,False
2,102,Goal mouth y co-ordinate,605,100.0%,True
3,231,GK Y Coordinate,605,100.0%,True
4,56,Zone,605,100.0%,True
...,...,...,...,...,...
69,487,*** NOT IN MAPPING ***,2,0.3%,False
70,358,*** NOT IN MAPPING ***,1,0.2%,False
71,484,*** NOT IN MAPPING ***,1,0.2%,False
72,122,Swerve Moving,1,0.2%,True


In [7]:
# ── Summary card ──────────────────────────────────────────────────────────────

all_qids_on_shots  = set(all_qual_counts.keys())
qids_in_mapping    = set(quals_mapping.keys())
unknown_qids       = all_qids_on_shots - qids_in_mapping
covered_qids       = all_qids_on_shots & qids_in_mapping

print("=" * 55)
print(" SHOT QUALIFIER AUDIT SUMMARY")
print("=" * 55)
print(f"  Shot events scanned          : {total_shots:,}")
print(f"  Distinct qualifier IDs seen  : {len(all_qids_on_shots)}")
print(f"  Covered by opta-qualifiers   : {len(covered_qids)}")
print(f"  ⚠  NOT in mapping (unknown) : {len(unknown_qids)}", end="")
print("  ← investigate!" if unknown_qids else "")
print("=" * 55)

if unknown_qids:
    print(f"\n⚠  Unknown qualifier IDs on shots: {sorted(unknown_qids)}")
    print("   See cell 5 above for full breakdown.")
else:
    print("\n✅  All qualifiers on shots are documented in opta-qualifiers.js.")

 SHOT QUALIFIER AUDIT SUMMARY
  Shot events scanned          : 5,793
  Distinct qualifier IDs seen  : 103
  Covered by opta-qualifiers   : 83
  ⚠  NOT in mapping (unknown) : 20  ← investigate!

⚠  Unknown qualifier IDs on shots: [300, 314, 343, 353, 358, 362, 374, 375, 390, 391, 395, 396, 458, 459, 468, 472, 474, 484, 487, 490]
   See cell 5 above for full breakdown.


In [8]:
# ── Video verification samples: qualifiers 328 and 428 on goals ──────────────
# For each qualifier: 10 goals WITH it, 10 goals WITHOUT it.
# Columns: source_match_id, period, minute, second — enough to locate the clip.

##TARGET_QUALIFIERS = [328, 458]
TARGET_QUALIFIERS = [24,25,26,96]
N_SAMPLES         = 10


def scan_goals_for_video(matches_dir: Path) -> list[dict]:
    """Re-scan all files for goals (typeId 16), pulling source_match_id and timeSec."""
    goals = []
    for fp in sorted(p for p in matches_dir.iterdir() if p.is_file() and not p.name.startswith(".")):
        try:
            data = extract_json_from_jsonp(fp)
        except Exception:
            continue
        source_match_id = data.get("matchInfo", {}).get("id", fp.name)
        for ev in data["liveData"]["event"]:
            if ev.get("typeId") != 16:
                continue
            qualifier_ids = [q.get("qualifierId") for q in ev.get("qualifier", [])]
            goals.append({
                "source_match_id": source_match_id,
                "period"         : ev.get("periodId"),
                "minute"         : ev.get("timeMin"),
                "second"         : ev.get("timeSec"),
                "qualifier_ids"  : qualifier_ids,
            })
    return goals


def show_goals(rows: list[dict], label: str) -> None:
    if not rows:
        print(f"  (no goals found for: {label})")
        return
    display(pd.DataFrame(rows)[["source_match_id", "period", "minute", "second", "qualifier_ids"]])


goals = scan_goals_for_video(MATCHES_DIR)
print(f"Total goals collected: {len(goals):,}\n")

for qid in TARGET_QUALIFIERS:
    with_q    = [g for g in goals if     qid in g["qualifier_ids"]][:N_SAMPLES]
    without_q = [g for g in goals if qid not in g["qualifier_ids"]][:N_SAMPLES]

    print("=" * 65)
    print(f"  qualifierId {qid}   —   {qual_name(qid, quals_mapping)}")
    print(f"  Goals WITH: {sum(1 for g in goals if qid in g['qualifier_ids'])}   |   "
          f"Goals WITHOUT: {sum(1 for g in goals if qid not in g['qualifier_ids'])}")
    print("=" * 65)

    print(f"\n  ✅  {N_SAMPLES} goals WITH qualifierId {qid}:")
    show_goals(with_q, f"WITH {qid}")

    print(f"\n  ❌  {N_SAMPLES} goals WITHOUT qualifierId {qid}:")
    show_goals(without_q, f"WITHOUT {qid}")
    print()


Total goals collected: 605

  qualifierId 24   —   Set piece
  Goals WITH: 28   |   Goals WITHOUT: 577

  ✅  10 goals WITH qualifierId 24:


,source_match_id,period,minute,second,qualifier_ids
0,11uxeay5z2isuekqd9fq0usyc,2,71,28,"[15, 17, 231, 328, 55, 154, 396, 29, 102, 230,..."
1,12lophscpllg367yx7yn0mpec,1,41,40,"[17, 103, 328, 78, 396, 55, 24, 395, 231, 374,..."
2,12lophscpllg367yx7yn0mpec,2,78,40,"[375, 103, 102, 458, 136, 16, 56, 395, 396, 23..."
3,15ir6pt6au1v00gtef02v4k,2,91,6,"[282, 231, 374, 24, 472, 214, 56, 77, 396, 230..."
4,18ojlro5jtdeu8nv72repur6c,2,86,25,"[395, 20, 16, 374, 24, 55, 214, 56, 375, 103, ..."
5,1eaz7mdny12eiuyeysnycnw2c,2,67,57,"[472, 396, 230, 281, 113, 458, 231, 77, 280, 2..."
6,1nc8632r4hii9bl1ygnappkpg,1,37,49,"[29, 230, 154, 102, 375, 328, 231, 24, 77, 103..."
7,1p93rklxg8b0b4dymsqjlcbh0,1,22,5,"[154, 29, 76, 72, 55, 231, 374, 396, 17, 375, ..."
8,1pmmfccm8cfhw6hua3w0gz28k,1,43,48,"[374, 108, 215, 396, 72, 458, 102, 395, 230, 1..."
9,1q09wu5t41r3rdtsv757pyf4k,2,59,19,"[230, 56, 374, 24, 55, 103, 29, 395, 396, 231,..."



  ❌  10 goals WITHOUT qualifierId 24:


,source_match_id,period,minute,second,qualifier_ids
0,10d1132abu0fa9xolj05top3o,2,74,13,"[20, 472, 16, 328, 214, 395, 396, 22, 458, 281..."
1,10qg7chtpoj0knqnvx2oduot0,2,63,52,"[458, 17, 280, 76, 374, 214, 396, 231, 390, 39..."
2,10qg7chtpoj0knqnvx2oduot0,2,76,31,"[328, 102, 29, 374, 20, 231, 17, 214, 396, 55,..."
3,10qg7chtpoj0knqnvx2oduot0,2,89,9,"[29, 17, 136, 374, 375, 231, 103, 154, 102, 39..."
4,113wccwt50kebz0thqf0fxkpg,1,8,28,"[22, 230, 102, 17, 395, 468, 72, 103, 231, 154..."
5,113wccwt50kebz0thqf0fxkpg,1,11,0,"[468, 231, 396, 328, 375, 20, 154, 214, 55, 22..."
6,113wccwt50kebz0thqf0fxkpg,1,41,28,"[374, 396, 215, 29, 20, 22, 102, 138, 103, 64,..."
7,113wccwt50kebz0thqf0fxkpg,2,60,3,"[79, 72, 113, 56, 215, 64, 231, 374, 395, 138,..."
8,11hplskmrac4dwfmqv72f9nh0,1,9,39,"[103, 56, 395, 230, 55, 231, 72, 375, 18, 22, ..."
9,11hplskmrac4dwfmqv72f9nh0,1,33,41,"[396, 55, 102, 328, 230, 56, 395, 121, 29, 375..."



  qualifierId 25   —   From corner
  Goals WITH: 70   |   Goals WITHOUT: 535

  ✅  10 goals WITH qualifierId 25:


...

,source_match_id,period,minute,second,qualifier_ids
0,1cg5aiu7phyw89gyr0m9hme50,2,81,57,"[102, 20, 328, 154, 120, 76, 230, 395, 18, 396..."
1,1d6oxum3q5ydyzsf48df1qas4,2,86,2,"[56, 282, 26, 458, 18, 472, 20, 103, 231, 280,..."
2,1r4k8mooxohtqr7vzjm29hul0,1,33,50,"[375, 472, 281, 282, 280, 231, 18, 56, 26, 122..."
3,1ts1vbi2mugv1401ifojdtqtw,2,68,3,"[374, 77, 18, 375, 458, 103, 26, 231, 281, 282..."
4,1wwdsnnj2m9ewz1d1mgt7y8k,1,36,30,"[26, 77, 230, 56, 18, 103, 102, 396, 375, 395,..."
5,du8jha9qxukb7ki17m3y3aj8,2,69,6,"[458, 18, 396, 26, 231, 395, 120, 81, 375, 374..."
6,fo7f0yq3c06q4hrw7vamnnro,2,48,29,"[121, 56, 395, 26, 458, 76, 375, 231, 72, 374,..."
7,m3rr5kwhsu6s770ey69xad5g,2,63,0,"[102, 26, 374, 77, 375, 18, 136, 396, 458, 103..."



  ❌  10 goals WITHOUT qualifierId 26:


,source_match_id,period,minute,second,qualifier_ids
0,10d1132abu0fa9xolj05top3o,2,74,13,"[20, 472, 16, 328, 214, 395, 396, 22, 458, 281..."
1,10qg7chtpoj0knqnvx2oduot0,2,63,52,"[458, 17, 280, 76, 374, 214, 396, 231, 390, 39..."
2,10qg7chtpoj0knqnvx2oduot0,2,76,31,"[328, 102, 29, 374, 20, 231, 17, 214, 396, 55,..."
3,10qg7chtpoj0knqnvx2oduot0,2,89,9,"[29, 17, 136, 374, 375, 231, 103, 154, 102, 39..."
4,113wccwt50kebz0thqf0fxkpg,1,8,28,"[22, 230, 102, 17, 395, 468, 72, 103, 231, 154..."
5,113wccwt50kebz0thqf0fxkpg,1,11,0,"[468, 231, 396, 328, 375, 20, 154, 214, 55, 22..."
6,113wccwt50kebz0thqf0fxkpg,1,41,28,"[374, 396, 215, 29, 20, 22, 102, 138, 103, 64,..."
7,113wccwt50kebz0thqf0fxkpg,2,60,3,"[79, 72, 113, 56, 215, 64, 231, 374, 395, 138,..."
8,11hplskmrac4dwfmqv72f9nh0,1,9,39,"[103, 56, 395, 230, 55, 231, 72, 375, 18, 22, ..."
9,11hplskmrac4dwfmqv72f9nh0,1,33,41,"[396, 55, 102, 328, 230, 56, 395, 121, 29, 375..."



  qualifierId 96   —   Corner situation
  Goals WITH: 0   |   Goals WITHOUT: 605

  ✅  10 goals WITH qualifierId 96:
  (no goals found for: WITH 96)

  ❌  10 goals WITHOUT qualifierId 96:


,source_match_id,period,minute,second,qualifier_ids
0,10d1132abu0fa9xolj05top3o,2,74,13,"[20, 472, 16, 328, 214, 395, 396, 22, 458, 281..."
1,10qg7chtpoj0knqnvx2oduot0,2,63,52,"[458, 17, 280, 76, 374, 214, 396, 231, 390, 39..."
2,10qg7chtpoj0knqnvx2oduot0,2,76,31,"[328, 102, 29, 374, 20, 231, 17, 214, 396, 55,..."
3,10qg7chtpoj0knqnvx2oduot0,2,89,9,"[29, 17, 136, 374, 375, 231, 103, 154, 102, 39..."
4,113wccwt50kebz0thqf0fxkpg,1,8,28,"[22, 230, 102, 17, 395, 468, 72, 103, 231, 154..."
5,113wccwt50kebz0thqf0fxkpg,1,11,0,"[468, 231, 396, 328, 375, 20, 154, 214, 55, 22..."
6,113wccwt50kebz0thqf0fxkpg,1,41,28,"[374, 396, 215, 29, 20, 22, 102, 138, 103, 64,..."
7,113wccwt50kebz0thqf0fxkpg,2,60,3,"[79, 72, 113, 56, 215, 64, 231, 374, 395, 138,..."
8,11hplskmrac4dwfmqv72f9nh0,1,9,39,"[103, 56, 395, 230, 55, 231, 72, 375, 18, 22, ..."
9,11hplskmrac4dwfmqv72f9nh0,1,33,41,"[396, 55, 102, 328, 230, 56, 395, 121, 29, 375..."


In [9]:
# ── Deep-dive: qualifierId 458 ────────────────────────────────────────────────

INSPECT_QID = 458

subset  = [s for s in shots if INSPECT_QID in s["qualifier_ids"]]
missing = [s for s in shots if INSPECT_QID not in s["qualifier_ids"]]
total   = len(shots)

print(f"qualifierId {INSPECT_QID}  —  {qual_name(INSPECT_QID, quals_mapping)}")
print(f"Appears on {len(subset):,} / {total:,} shot events ({len(subset)/total*100:.1f}%)\n")

# ── 0. Does qualifier 458 appear on non-shot events? ─────────────────────────
print("── 0. All event types carrying qualifierId 458 (full match scan) ───")
counts_by_type: Counter = Counter()
for fp in sorted(p for p in MATCHES_DIR.iterdir() if p.is_file() and not p.name.startswith(".")):
    try:
        data = extract_json_from_jsonp(fp)
    except Exception:
        continue
    for ev in data["liveData"]["event"]:
        qids = [q.get("qualifierId") for q in ev.get("qualifier", [])]
        if INSPECT_QID in qids:
            counts_by_type[ev.get("typeId")] += 1

KNOWN_TYPE_NAMES = {
    1: "Pass", 2: "Offside pass", 3: "Take on", 4: "Foul", 5: "Out",
    6: "Corner awarded", 7: "Tackle", 8: "Interception", 10: "Save",
    11: "Claim", 12: "Clearance", 13: "Miss", 14: "Post",
    15: "Attempt Saved", 16: "Goal", 17: "Card", 18: "Player off",
    19: "Player on", 32: "Start period", 34: "Team set up",
    40: "Other ball contact", 49: "Ball recovery", 55: "Ball touch",
    61: "Chance missed", -1: "Carry (synthetic)",
}

total_all = sum(counts_by_type.values())
print(f"  Events with qualifier 458 across ALL types: {total_all:,}\n")
rows_type = []
for tid, cnt in sorted(counts_by_type.items(), key=lambda x: -x[1]):
    rows_type.append({
        "typeId"    : tid,
        "type_name" : KNOWN_TYPE_NAMES.get(tid, f"typeId {tid}"),
        "count"     : cnt,
        "pct_of_all": f"{cnt/total_all*100:.1f}%",
        "is_shot"   : tid in SHOT_TYPE_IDS,
    })
display(pd.DataFrame(rows_type))

# ── 1. Which shot typeIds carry it? ──────────────────────────────────────────
print("\n── 1. Shot type breakdown ──────────────────────────────────────────")
type_dist = Counter(f"{s['type_id']} ({s['type_name']})" for s in subset)
for label, cnt in type_dist.most_common():
    print(f"  {cnt:>4}x  ({cnt/len(subset)*100:.1f}%)  {label}")

# ── 2. Qualifier value distribution ──────────────────────────────────────────
print("\n── 2. Qualifier value distribution ─────────────────────────────────")
val_dist = Counter(s["qualifier_vals"].get(INSPECT_QID) for s in subset)
for val, cnt in val_dist.most_common(15):
    print(f"  {cnt:>4}x  {repr(val)}")

# ── 3. Top co-occurring qualifiers ───────────────────────────────────────────
print("\n── 3. Top co-occurring qualifiers (on shots WITH 458) ──────────────")
cooccur: Counter = Counter()
for s in subset:
    for other in s["qualifier_ids"]:
        if other != INSPECT_QID:
            cooccur[other] += 1
for other_qid, cnt in cooccur.most_common(20):
    print(f"  {cnt:>4}x  ({cnt/len(subset)*100:.1f}%)  qualifierId {other_qid:>4}  —  {qual_name(other_qid, quals_mapping)}")

# ── 4. Companion analysis: P(458 | qualifier X) vs P(458 | ¬qualifier X) ─────
# Each row answers: "of all shot events that carry qualifier X, what % also have 458?"
# This is the reverse of the old table, which asked "of 458-events, how many have X?"
# Example reading: P(458 | Penalty) = 100% means every penalty shot carries 458.
# Baseline: 458 appears on ~25.9% of all shots, so lift > 1 means X predicts 458.
print("\n── 4. Companion analysis ────────────────────────────────────────────")
print(f"  Reading: 'P(458|X)' = among shots WITH qualifier X, % that also carry 458")
print(f"  Baseline P(458) = {len(subset)/total*100:.1f}%  (458 appears on this share of all shots)\n")

cooccur_missing: Counter = Counter()
for s in missing:
    for other in s["qualifier_ids"]:
        cooccur_missing[other] += 1

rows = []
all_qids = set(cooccur.keys()) | set(cooccur_missing.keys())
for qid in all_qids:
    cnt_with_both = cooccur.get(qid, 0)        # has X  AND has 458
    cnt_x_no_458  = cooccur_missing.get(qid, 0) # has X  but  no 458
    cnt_x_total   = cnt_with_both + cnt_x_no_458

    cnt_458_no_x  = len(subset) - cnt_with_both  # no X  but has 458
    cnt_no_x      = total - cnt_x_total           # no X  (with or without 458)

    pct_458_if_x    = cnt_with_both / cnt_x_total * 100 if cnt_x_total > 0 else 0.0
    pct_458_if_no_x = cnt_458_no_x  / cnt_no_x   * 100 if cnt_no_x    > 0 else 0.0
    lift = pct_458_if_x / pct_458_if_no_x if pct_458_if_no_x > 0 else float("inf")

    rows.append({
        "qualifierId"  : qid,
        "name"         : qual_name(qid, quals_mapping),
        "shots_with_X" : cnt_x_total,
        "P(458|X)"     : f"{pct_458_if_x:.1f}%",
        "P(458|no X)"  : f"{pct_458_if_no_x:.1f}%",
        "lift"         : round(lift, 2),
    })

df_lift = (
    pd.DataFrame(rows)
    .sort_values("lift", ascending=False)
    .reset_index(drop=True)
)
display(df_lift.head(25))

# ── 5. Outcome distribution ───────────────────────────────────────────────────
print("\n── 5. Outcome distribution ─────────────────────────────────────────")
out_dist = Counter(s["outcome"] for s in subset)
for val, cnt in out_dist.most_common():
    print(f"  outcome={val}  {cnt:>4}x  ({cnt/len(subset)*100:.1f}%)")

# ── 6. Sample rows ────────────────────────────────────────────────────────────
print(f"\n── 6. Sample events with qualifierId {INSPECT_QID} (up to 15) ────────")
display(
    pd.DataFrame(subset[:15])[[
        "match", "type_name", "period", "minute", "outcome", "x", "y", "qualifier_ids"
    ]]
)

qualifierId 458  —  *** NOT IN MAPPING ***
Appears on 1,502 / 5,793 shot events (25.9%)

── 0. All event types carrying qualifierId 458 (full match scan) ───
  Events with qualifier 458 across ALL types: 1,522



,typeId,type_name,count,pct_of_all,is_shot
0,15,Attempt Saved,752,49.4%,True
1,13,Miss,538,35.3%,True
2,16,Goal,184,12.1%,True
3,14,Post,28,1.8%,True
4,43,typeId 43,10,0.7%,False
5,84,typeId 84,10,0.7%,False



── 1. Shot type breakdown ──────────────────────────────────────────
   752x  (50.1%)  15 (Attempt Saved)
   538x  (35.8%)  13 (Miss)
   184x  (12.3%)  16 (Goal)
    28x  (1.9%)  14 (Post)

── 2. Qualifier value distribution ─────────────────────────────────
  1502x  None

── 3. Top co-occurring qualifiers (on shots WITH 458) ──────────────
  1502x  (100.0%)  qualifierId  103  —  Goal mouth z co-ordinate
  1502x  (100.0%)  qualifierId  102  —  Goal mouth y co-ordinate
  1502x  (100.0%)  qualifierId   56  —  Zone
  1204x  (80.2%)  qualifierId  231  —  GK Y Coordinate
  1204x  (80.2%)  qualifierId  230  —  GK X Coordinate
   870x  (57.9%)  qualifierId  146  —  Blocked x co-ordinate
   870x  (57.9%)  qualifierId  147  —  Blocked y co-ordinate
   847x  (56.4%)  qualifierId   20  —  Right footed
   787x  (52.4%)  qualifierId   22  —  Regular play
   724x  (48.2%)  qualifierId  233  —  Opposite related event ID
   720x  (47.9%)  qualifierId  328  —  First Time
   689x  (45.9%)  qualifierId 

,qualifierId,name,shots_with_X,P(458|X),P(458|no X),lift
0,102,Goal mouth y co-ordinate,5793,25.9%,0.0%,inf
1,56,Zone,5793,25.9%,0.0%,inf
2,103,Goal mouth z co-ordinate,5793,25.9%,0.0%,inf
3,9,Penalty,85,100.0%,24.8%,4.03
4,390,*** NOT IN MAPPING ***,85,100.0%,24.8%,4.03
5,353,*** NOT IN MAPPING ***,85,100.0%,24.8%,4.03
6,26,Free kick,150,96.7%,24.0%,4.02
7,487,*** NOT IN MAPPING ***,3,100.0%,25.9%,3.86
8,280,Fantasy Assist Type,116,88.8%,24.6%,3.60
9,281,Fantasy Assisted By,116,88.8%,24.6%,3.60



── 5. Outcome distribution ─────────────────────────────────────────
  outcome=1  1502x  (100.0%)

── 6. Sample events with qualifierId 458 (up to 15) ────────


,match,type_name,period,minute,outcome,x,y,qualifier_ids
0,10d1132abu0fa9xolj05top3o,Miss,1,4,1,77.2,55.8,"[328, 103, 215, 231, 102, 160, 230, 77, 72, 45..."
1,10d1132abu0fa9xolj05top3o,Attempt Saved,1,34,1,74.3,35.8,"[102, 328, 215, 103, 24, 146, 20, 82, 18, 233,..."
2,10d1132abu0fa9xolj05top3o,Attempt Saved,1,47,1,92.9,32.9,"[56, 102, 20, 233, 22, 147, 103, 78, 458, 328,..."
3,10d1132abu0fa9xolj05top3o,Attempt Saved,2,54,1,90.6,39.3,"[22, 15, 102, 328, 233, 56, 147, 458, 146, 103..."
4,10d1132abu0fa9xolj05top3o,Attempt Saved,2,61,1,90.2,48.5,"[146, 458, 147, 108, 103, 231, 233, 25, 17, 10..."
5,10d1132abu0fa9xolj05top3o,Attempt Saved,2,61,1,79.9,41.5,"[231, 102, 146, 458, 147, 215, 82, 230, 18, 56..."
6,10d1132abu0fa9xolj05top3o,Attempt Saved,2,74,1,92.0,42.7,"[146, 15, 214, 147, 102, 56, 22, 233, 328, 17,..."
7,10d1132abu0fa9xolj05top3o,Goal,2,74,1,96.2,54.6,"[20, 472, 16, 328, 214, 395, 396, 22, 458, 281..."
8,10d1132abu0fa9xolj05top3o,Attempt Saved,2,80,1,76.6,31.1,"[18, 102, 146, 26, 103, 56, 458, 78, 147, 120,..."
9,10d1132abu0fa9xolj05top3o,Attempt Saved,2,92,1,96.8,69.5,"[20, 458, 147, 146, 22, 233, 102, 468, 103, 65..."
